# Paper 1 (brain paper): analyses

Everything uses the released connectomes as they are (raw streamline counts, native density, no thresholding or harmonisation) and the memory capacity (MC) of the simulated echo-state reservoir (C++ engine in `ESNcpp/`, input scale $10^{-5}$, $\rho=0.99$). No closed form is used here. Results go to `paper/data/p1/`; simulations are cached in `paper/data/p1/sim/`, so re-running only recomputes what is missing.

1. Walk orders: how much each order of the walk expansion reproduces subgraph centrality, node by node, and predicts MC.
2. Predictors of MC with an a priori link to the expansion or to the literature on linear dynamics on connectomes.
3. Null models (10 realisations per stochastic null).
4. Interhemispheric weight across the lifespan (equalised in every connectome, simulated) and CALM referred children (simulated).
5. Development, stated honestly: within-cohort age trends and the camCAN turning point.

In [ ]:
import json, math, os, sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

REPO = Path.cwd().resolve().parent                    # this notebook lives in paper/
sys.path.insert(0, str(REPO / "chain"))
import chain_analysis as ca                            # loaders and runners of the C++ echo-state engine (ESNcpp)

# Paths to the data: edit to your machine (or set NETLEARN_DATA)
DATA_DIR = Path(os.environ.get("NETLEARN_DATA", "~/netlearn-data")).expanduser()   # export of the 3,901 connectomes
ALLMAT = DATA_DIR / "all_data.mat"                     # original release (CALM referred children)
DEMO = DATA_DIR / "demographics_all.csv"
OUT = REPO / "paper/data/p1"; WORK = OUT / "sim"; WORK.mkdir(parents=True, exist_ok=True)
ESN = dict(ca.ESN_CFG, input_scale=1e-5, log_state=0, log_theory=0, threads=os.cpu_count())   # the reservoir of the paper

sid, mats, meta = ca.load_export(str(DATA_DIR)); sid = np.asarray(sid).astype(int)
d = pd.read_csv(REPO / "paper/data/v3/subjects_v3.csv").set_index("sid").loc[sid]   # simulated MC, age, cohort, structure
N = {}

## Parcellations (labels of each released cohort) and symmetric balancing

In [ ]:
AAL90 = """Precentral_L Precentral_R Frontal_Sup_L Frontal_Sup_R Frontal_Sup_Orb_L Frontal_Sup_Orb_R Frontal_Mid_L
Frontal_Mid_R Frontal_Mid_Orb_L Frontal_Mid_Orb_R Frontal_Inf_Oper_L Frontal_Inf_Oper_R Frontal_Inf_Tri_L Frontal_Inf_Tri_R
Frontal_Inf_Orb_L Frontal_Inf_Orb_R Rolandic_Oper_L Rolandic_Oper_R Supp_Motor_Area_L Supp_Motor_Area_R Olfactory_L
Olfactory_R Frontal_Sup_Medial_L Frontal_Sup_Medial_R Frontal_Med_Orb_L Frontal_Med_Orb_R Rectus_L Rectus_R Insula_L
Insula_R Cingulum_Ant_L Cingulum_Ant_R Cingulum_Mid_L Cingulum_Mid_R Cingulum_Post_L Cingulum_Post_R Hippocampus_L
Hippocampus_R ParaHippocampal_L ParaHippocampal_R Amygdala_L Amygdala_R Calcarine_L Calcarine_R Cuneus_L Cuneus_R
Lingual_L Lingual_R Occipital_Sup_L Occipital_Sup_R Occipital_Mid_L Occipital_Mid_R Occipital_Inf_L Occipital_Inf_R
Fusiform_L Fusiform_R Postcentral_L Postcentral_R Parietal_Sup_L Parietal_Sup_R Parietal_Inf_L Parietal_Inf_R
SupraMarginal_L SupraMarginal_R Angular_L Angular_R Precuneus_L Precuneus_R Paracentral_Lobule_L Paracentral_Lobule_R
Caudate_L Caudate_R Putamen_L Putamen_R Pallidum_L Pallidum_R Thalamus_L Thalamus_R Heschl_L Heschl_R Temporal_Sup_L
Temporal_Sup_R Temporal_Pole_Sup_L Temporal_Pole_Sup_R Temporal_Mid_L Temporal_Mid_R Temporal_Pole_Mid_L
Temporal_Pole_Mid_R Temporal_Inf_L Temporal_Inf_R""".split()
AAL3_94 = """Precentral_L Precentral_R Frontal_Sup_2_L Frontal_Sup_2_R Frontal_Mid_2_L Frontal_Mid_2_R Frontal_Inf_Oper_L
Frontal_Inf_Oper_R Frontal_Inf_Tri_L Frontal_Inf_Tri_R Frontal_Inf_Orb_2_L Frontal_Inf_Orb_2_R Rolandic_Oper_L Rolandic_Oper_R
Supp_Motor_Area_L Supp_Motor_Area_R Olfactory_L Olfactory_R Frontal_Sup_Medial_L Frontal_Sup_Medial_R Frontal_Med_Orb_L
Frontal_Med_Orb_R Rectus_L Rectus_R OFCmed_L OFCmed_R OFCant_L OFCant_R OFCpost_L OFCpost_R OFClat_L OFClat_R Insula_L
Insula_R EMPTY35 EMPTY36 Cingulate_Mid_L Cingulate_Mid_R Cingulate_Post_L Cingulate_Post_R Hippocampus_L
Hippocampus_R ParaHippocampal_L ParaHippocampal_R Amygdala_L Amygdala_R Calcarine_L Calcarine_R Cuneus_L Cuneus_R Lingual_L
Lingual_R Occipital_Sup_L Occipital_Sup_R Occipital_Mid_L Occipital_Mid_R Occipital_Inf_L Occipital_Inf_R Fusiform_L
Fusiform_R Postcentral_L Postcentral_R Parietal_Sup_L Parietal_Sup_R Parietal_Inf_L Parietal_Inf_R SupraMarginal_L
SupraMarginal_R Angular_L Angular_R Precuneus_L Precuneus_R Paracentral_Lobule_L Paracentral_Lobule_R Caudate_L Caudate_R
Putamen_L Putamen_R Pallidum_L Pallidum_R EMPTY81 EMPTY82 Heschl_L Heschl_R Temporal_Sup_L Temporal_Sup_R
Temporal_Pole_Sup_L Temporal_Pole_Sup_R Temporal_Mid_L Temporal_Mid_R Temporal_Pole_Mid_L Temporal_Pole_Mid_R
Temporal_Inf_L Temporal_Inf_R""".split()
AAL3 = [n for n in AAL3_94 if not n.startswith("EMPTY")]
assert len(AAL90) == 90 and len(AAL3) == 90
AAL3_COHORTS = {6, 7, 8, 9}                      # HCPd, HCPya, camCAN, HCPa
CANON = lambda n: n.replace("_2_", "_").replace("Cingulate", "Cingulum")
SHARED = [n for n in AAL90 if n in [CANON(m) for m in AAL3]]           # 82 regions
MEDIAL = ("Supp_Motor_Area", "Paracentral_Lobule", "Frontal_Sup_Medial", "Precuneus", "Cuneus", "Calcarine",
          "Lingual", "Cingulum", "Cingulate", "Frontal_Med_Orb", "Rectus", "Olfactory", "OFCmed")


def labels(ds):
    return AAL3 if int(ds) in AAL3_COHORTS else AAL90


def edge_sets(ds):
    """Index arrays of homotopic (medial / lateral) and all interhemispheric pairs for a cohort's atlas."""
    L = labels(ds); idx = {n: i for i, n in enumerate(L)}
    hom_med, hom_lat = [], []
    for n, i in idx.items():
        if n.endswith("_L") and n[:-2] + "_R" in idx:
            (hom_med if n.startswith(MEDIAL) else hom_lat).append((i, idx[n[:-2] + "_R"]))
    left = np.array([n.endswith("_L") for n in L])
    return np.array(hom_med), np.array(hom_lat), np.logical_xor.outer(left, left)


EDGES = {a: edge_sets(6 if a == 3 else 1) for a in (1, 3)}   # 1: AAL90, 3: AAL3


def sets_of(ds):
    return EDGES[3 if int(ds) in AAL3_COHORTS else 1]


def balance(A, tol=1e-10, iters=200000):
    """Symmetric Sinkhorn-Knopp: D A D with all strengths equal to the mean strength of A."""
    on = A.sum(1) > 0; B = A[np.ix_(on, on)]; target = B.sum() / B.shape[0]; x = np.ones(B.shape[0])
    for _ in range(iters):
        xn = np.sqrt(x * target / (B @ x))
        if np.max(np.abs(xn - x)) < tol * xn.max():
            x = xn; break
        x = xn
    out = np.zeros_like(A); out[np.ix_(on, on)] = B * np.outer(x, x)
    return out if np.all(np.isfinite(out)) and np.allclose(out.sum(1)[on], target, rtol=1e-4) else None

In [ ]:
def simulate(name, ids, matrices, win_reps=1):
    """MC of each matrix with the C++ echo-state engine (same settings as the 3,901 connectomes); cached."""
    out = WORK / f"mc_{name}.csv"
    if not out.exists():
        csv = WORK / f"{name}.csv"; ca.write_connectomes_csv(csv, ids, matrices)
        ca.run_cfg(REPO / "ESNcpp/esn_mc", WORK / f"cfg_{name}.txt", {**ESN, "win_reps": win_reps, "in_csv": str(csv), "out_csv": str(out)})
        csv.unlink()
    return pd.read_csv(out).rename(columns={"MC_Glob": "MC"}).set_index("subject_id").MC


def r_(a, b):
    return float(np.corrcoef(a, b)[0, 1])


LOGR = np.log(d.rs.values); B_LOG = float(np.polyfit(LOGR, d.MC.values, 1)[0])   # slope of MC on log stable rank

## 1–2. Walk orders and predictors of memory

All quantities on $\tilde W=0.99\,W/\rho(W)$, with eigenvalues $\lambda_k$ and eigenvectors $v_k$. Single orders: $\mathrm{Tr}\,\tilde W^n$. Subgraph centrality truncated at order $n$: $1+\sum_{m=2}^{n}\mathrm{Tr}\,\tilde W^m/(m!\,N)$. Literature metrics: communicability $e^{\tilde W}$ (Estrada; Crofts & Higham), linear attenuation $(I-\alpha\tilde W)^{-1}$ and the covariance of the spatial autoregressive model $(I-\alpha\tilde W)^{-2}$ (Fakhar et al. 2025; Zamora-López et al.), average controllability $\langle(I-\tilde W^2)^{-1}\rangle_{ii}$, the trace of the controllability Gramian (Gu et al. 2015), and the eigenvalue entropy ("structural complexity", Fakhar et al. 2026).

In [ ]:
ALPHA = (0.25, 0.5, 0.75, 0.9, 0.95, 0.99)
kb = np.array([1, 3, 6, 10, 15, 20, 25, 30, 35, 40, 50, 90]); cnt = np.zeros(len(kb) - 1); acc = {n: np.zeros(len(kb) - 1) for n in (2, 3, 4, 5)}
SUM = {key: {q: np.zeros(len(kb) - 1) for q in ("x", "xx", "xy")} for key in ("Y", "s2", "ord2", "ord3", "ord4", "ord5")}
SC = {q: np.zeros(len(kb) - 1) for q in ("y", "yy")}
rows, node_rows = [], []; NODE_SUBJ = set(np.random.default_rng(1).choice(sid, 150, replace=False))   # nodes shown in the C_ii figure
for s, A in zip(sid, mats):
    A = A.astype(float); lam, V = np.linalg.eigh(A); rho = np.abs(lam).max(); lt = 0.99 * lam / rho; Wt = 0.99 * A / rho; n_ = len(lt)
    u = V.sum(0) ** 2                                  # weight of each mode in sums over node pairs
    off = lambda f: (float(u @ f) - float(f.sum())) / (n_ * (n_ - 1))
    k = (A > 0).sum(1); st = Wt.sum(1); p = np.abs(lt) / np.abs(lt).sum()
    An, Ab = Wt, (A > 0).astype(float)                  # weighted (Zhang-Horvath on the normalised W, the c^w of Eq. zh) and binary clustering
    tw, dw = ((An @ An) * An).sum(1), An.sum(1) ** 2 - (An ** 2).sum(1); tb, db = ((Ab @ Ab) * Ab).sum(1), k * (k - 1.0)
    rows.append(dict(sid=int(s), k=k.mean(), clust_w=float((tw[dw > 0] / dw[dw > 0]).mean()), clust_b=float((tb[db > 0] / db[db > 0]).mean()), s_raw=float(A.sum(1).mean()), G=float((st ** 2).mean()), rs=float((lt ** 2).sum() / 0.9801),
                     cii=float(np.exp(lt).mean()), comm=off(np.exp(lt)), lam=off(1 / (1 - 0.5 * lt)),
                     avgctrl=float((1 / (1 - lt ** 2)).mean()), spec_entropy=float(-(p * np.log(p)).sum() / np.log(n_)),
                     **{f"tr{m}": float((lt ** m).sum()) for m in range(2, 9)},
                     **{f"cii_ord{m}": 1 + sum(float((lt ** j).sum()) / math.factorial(j) for j in range(2, m + 1)) / n_ for m in range(2, 9)},
                     **{f"sar{a}": float((1 / (1 - a * lt) ** 2).mean()) for a in ALPHA}))
    W2 = Wt @ Wt; W3 = W2 @ Wt; W4 = W3 @ Wt; W5 = W4 @ Wt; Cii = (V ** 2) @ np.exp(lt)
    T = {2: np.diag(W2) / 2, 3: np.diag(W3) / 6, 4: np.diag(W4) / 24, 5: np.diag(W5) / 120}
    if s in NODE_SUBJ:
        node_rows.append(pd.DataFrame(dict(sid=int(s), k=k, C=Cii, ord2=1 + T[2], ord3=1 + T[2] + T[3], ord4=1 + T[2] + T[3] + T[4], ord5=1 + T[2] + T[3] + T[4] + T[5])))
    ok = k > 0; b = np.clip(np.searchsorted(kb, k[ok], side="right") - 1, 0, len(kb) - 2); np.add.at(cnt, b, 1)
    for m in (2, 3, 4, 5):
        np.add.at(acc[m], b, sum(T[j] for j in range(2, m + 1))[ok] / (Cii[ok] - 1))
    Yd = np.where(st > 0, (Wt ** 2).sum(1) / np.where(st > 0, st, 1) ** 2, 0.0)
    approx = {"Y": Yd, "s2": st ** 2, "ord2": 1 + T[2], "ord3": 1 + T[2] + T[3], "ord4": 1 + T[2] + T[3] + T[4], "ord5": 1 + T[2] + T[3] + T[4] + T[5]}
    for key, a in approx.items():
        np.add.at(SUM[key]["x"], b, a[ok]); np.add.at(SUM[key]["xx"], b, a[ok] ** 2); np.add.at(SUM[key]["xy"], b, a[ok] * Cii[ok])
    np.add.at(SC["y"], b, Cii[ok]); np.add.at(SC["yy"], b, Cii[ok] ** 2)
race = pd.DataFrame(rows).set_index("sid").join(d[["MC", "age", "cohort", "Ratio", "L", "hom_F2"]])
race.to_csv(OUT / "race_metrics.csv")
orders = pd.DataFrame(dict(k_lo=kb[:-1], k_hi=kb[1:], n_nodes=cnt, **{f"frac{m}": acc[m] / cnt for m in (2, 3, 4, 5)}))
orders.to_csv(OUT / "cii_orders_by_degree.csv", index=False)
def corr_from_sums(n, x, xx, xy, y, yy):
    return (n * xy - x * y) / np.sqrt((n * xx - x ** 2) * (n * yy - y ** 2))
r2 = pd.DataFrame(dict(k_lo=kb[:-1], k_hi=kb[1:], n_nodes=cnt, **{f"r2_{key}": corr_from_sums(cnt, v["x"], v["xx"], v["xy"], SC["y"], SC["yy"]) ** 2
                                                                     for key, v in SUM.items()}))
r2.to_csv(OUT / "cii_r2_by_degree.csv", index=False)
N["cii_r2_all"] = {key: float(corr_from_sums(cnt.sum(), v["x"].sum(), v["xx"].sum(), v["xy"].sum(), SC["y"].sum(), SC["yy"].sum()) ** 2) for key, v in SUM.items()}
nodes = pd.concat(node_rows); nodes = nodes[nodes.k > 0]; nodes.to_csv(OUT / "cii_nodes_sample.csv", index=False)
N["cii_nodes"] = dict(n=len(nodes), R2_order2=r_(nodes.C, nodes.ord2) ** 2, R2_order3=r_(nodes.C, nodes.ord3) ** 2)
orders.round(4)

In [ ]:
# correlation with MC: pooled, within each cohort (median, range) and given the stable rank (cubic in log r_s)
X = np.c_[np.ones(len(race)), LOGR, LOGR ** 2, LOGR ** 3]
res = lambda y: y - X @ np.linalg.lstsq(X, y, rcond=None)[0]
LOG = {"G", "L", "rs", "avgctrl", "lam", "s_raw", "clust_w", "clust_b"} | {f"sar{a}" for a in ALPHA} | {f"tr{m}" for m in range(2, 9)}
def tr(df, c): return np.log(df[c]) if c in LOG else df[c]
N["race"] = {}
for c in ["k", "G", "Ratio", "L", "rs", "tr3", "cii", "comm", "lam", "sar0.5", "avgctrl", "spec_entropy", "clust_w", "clust_b"] + \
         [f"tr{m}" for m in range(2, 9)] + [f"cii_ord{m}" for m in range(2, 9)] + [f"sar{a}" for a in ALPHA]:
    w = [r_(tr(g, c), g.MC) for _, g in race.groupby("cohort")]
    N["race"][c] = dict(r=r_(tr(race, c), race.MC), partial=r_(res(tr(race, c).values), res(race.MC.values)),
                        within_median=float(np.median(w)), within_min=float(min(w)), within_max=float(max(w)))
N["R2_MC_logrs"] = r_(LOGR, race.MC) ** 2
def age_given_rs(g):                                    # correlation of MC with age given the stable rank (cubic in log r_s), within a regime
    x = np.log(g.rs.values); Z = np.c_[np.ones(len(x)), x, x ** 2, x ** 3]; rr = lambda y: y - Z @ np.linalg.lstsq(Z, y, rcond=None)[0]
    return r_(rr(g.age.values.astype(float)), rr(g.MC.values))
N["age_given_rs"] = {reg: age_given_rs(g) for reg, g in (("development", race[race.age <= 32]), ("aging", race[race.age > 32]))}
N["race_regime"] = {c: {reg: r_(tr(g, c), g.MC) for reg, g in (("development", race[race.age <= 32]), ("aging", race[race.age > 32]))}
                    for c in ("rs", "cii", "comm", "tr3", "tr4", "tr5", "tr6", "tr7", "tr8", "Ratio", "s_raw", "k", "avgctrl", "clust_w", "clust_b")}
N["race"]["s_raw"] = dict(r=r_(tr(race, "s_raw"), race.MC), partial=r_(res(tr(race, "s_raw").values), res(race.MC.values)))
pd.DataFrame(N["race"]).T.round(3)

## Homotopic share on the 82 regions shared by both parcellations (robustness of $h$ across atlases)

In [ ]:
h82 = []
for s, A in zip(sid, mats):
    names = [CANON(n) for n in labels(int(d.loc[s, "dataset"]))]; ix = [names.index(n) for n in SHARED]; B = A[np.ix_(ix, ix)].astype(float)
    pairs = [(i, SHARED.index(n[:-2] + "_R")) for i, n in enumerate(SHARED) if n.endswith("_L") and n[:-2] + "_R" in SHARED]
    h82.append(2 * sum(B[i, j] ** 2 for i, j in pairs) / (B ** 2).sum())
d["h82"] = h82; N["h_vs_h82"] = r_(d.hom_F2, d.h82)
print("r(h, h on 82 shared regions) =", round(N["h_vs_h82"], 4))

## 3. Null models (simulated; 10 realisations per stochastic null)

The 100 age-balanced connectomes of v1. Uniform, broken stick and reshuffle (symmetric) are generated inside `esn_surrogates` (10 realisations each, Real re-simulated in every realisation). Interhemispheric ×½ is deterministic and simulated with `esn_mc` averaging over 10 input realisations, with the same seeds as its real network. The predicted change is $b\,\log(r_s^{null}/r_s)$, with $b$ the slope of MC on $\log r_s$ across the 3,901 connectomes.

In [ ]:
pick = np.array(sorted(pd.read_csv(REPO / "paper/data/surrogates.csv").subject_id.unique()))
A100 = np.array([mats[np.where(sid == s)[0][0]].astype(float) for s in pick])
surr_csv = WORK / "surr10.csv"
if not surr_csv.exists():
    ca.write_connectomes_csv(WORK / "real100.csv", pick, A100)
    cfg = {k: v for k, v in ca.ESN_CFG.items() if k not in ("steps", "tau", "win_reps")}
    ca.run_cfg(REPO / "ESNcpp/esn_surrogates", WORK / "cfg_surr.txt", {**cfg, **ca.SURR_CFG, "n_real": 10, "input_scale": 1e-5,
               "log_theory": 1, "asym_control": 0, "threads": os.cpu_count(), "in_csv": str(WORK / "real100.csv"), "out_csv": str(surr_csv)})
surr = pd.read_csv(surr_csv)                            # m2 = Tr(W~^2)/N of each simulated matrix; MC_lin is not used
real = surr[surr.model == "Real"].set_index(["subject_id", "realization"])
rows = []
for m in ("Uniform", "BrokenStick", "Reshuffle"):
    g = surr[surr.model == m].set_index(["subject_id", "realization"])
    dd = pd.DataFrame(dict(dMC=g.MC - real.MC, pred=B_LOG * np.log(g.m2 / real.m2))).groupby(level=0).mean()
    rows += [dict(sid=int(s), model=m, dMC=r.dMC, dMC_pred=r.pred) for s, r in dd.iterrows()]
rs_of = lambda A: float((A ** 2).sum() / np.abs(np.linalg.eigvalsh(A)).max() ** 2)
Ahalf = np.array([A * np.where(sets_of(int(d.loc[s, "dataset"]))[2], 0.5, 1.0) for s, A in zip(pick, A100)])
mc_r = simulate("null_real100", pick, A100, win_reps=10); mc_n = simulate("null_InterHalf100", pick, Ahalf, win_reps=10)
rows += [dict(sid=int(s), model="InterHalf", dMC=float(mc_n.loc[s] - mc_r.loc[s]), dMC_pred=B_LOG * np.log(rs_of(An) / rs_of(Ar)))
         for s, An, Ar in zip(pick, Ahalf, A100)]
nulls = pd.DataFrame(rows); nulls.to_csv(OUT / "nulls_sim.csv", index=False)
p = {m: stats.wilcoxon(g.dMC).pvalue for m, g in nulls.groupby("model")}
order = np.argsort(list(p.values())); padj = np.minimum.accumulate((np.array(list(p.values()))[order] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
pfdr = dict(zip(np.array(list(p))[order], np.minimum(padj, 1)))
N["nulls"] = {m: dict(n=len(g), dMC_median=float(g.dMC.median()), dMC_pred_median=float(g.dMC_pred.median()),
                      r_obs_pred=r_(g.dMC, g.dMC_pred), p=float(p[m]), p_fdr=float(pfdr[m])) for m, g in nulls.groupby("model")}
pd.DataFrame(N["nulls"]).T.round(3)

## 4a. Interhemispheric weight across the lifespan (simulated)

Node level: share of each region's squared weight carried by its homotopic connection, by degree (null model: 1/k if the weights of a region were exchangeable). Lifespan: every connectome gets the same interhemispheric weight fraction (the pooled mean), by multiplying all its interhemispheric weights by one factor, and its MC is simulated; the same with the homotopic connections only. Aging cohorts (camCAN and HCPa, age > 32) are pooled where stated: they agree at matched ages.

In [ ]:
ds = d.dataset.values.astype(int)
def hom_mask(dd):
    hm, hl, _ = sets_of(dd); M = np.zeros((90, 90), bool)
    for a_, b_ in np.r_[hm, hl]: M[a_, b_] = M[b_, a_] = True
    return M
HM = {dd: hom_mask(dd) for dd in np.unique(ds)}
rows = []
for i in range(len(mats)):
    A = mats[i].astype(float); k = (A > 0).sum(1); q = (A ** 2).sum(1)
    for a_, b_ in zip(*np.where(np.triu(HM[ds[i]]))):
        for x, y in ((a_, b_), (b_, a_)):
            if A[x, y] > 0: rows.append((k[x], A[x, y] ** 2 / q[x]))
hk = pd.DataFrame(rows, columns=["k", "share"])
hk.groupby("k").share.describe(percentiles=[0.25, 0.5, 0.75])[["count", "mean", "std", "25%", "50%", "75%"]].to_csv(OUT / "node_homotopic_share.csv")
N["node_homotopic"] = dict(mean_share_k18_45=float(hk[(hk.k >= 18) & (hk.k <= 45)].share.mean()), mean_null_k18_45=float((1 / hk[(hk.k >= 18) & (hk.k <= 45)].k).mean()))

fr = np.array([mats[i][sets_of(ds[i])[2]].sum() / mats[i].sum() for i in range(len(mats))]); F0 = float(fr.mean())
Aeq = []
for i in range(len(mats)):
    A = mats[i].astype(float); inter = sets_of(ds[i])[2]; I = A[inter].sum(); T = A.sum()
    Aeq.append(A * np.where(inter, F0 * (T - I) / (I * (1 - F0)), 1.0))
d["MC_eq"] = simulate("inter_equalised", sid, np.array(Aeq)).loc[sid].values
d["rs_eq"] = [float((A ** 2).sum() / np.abs(np.linalg.eigvalsh(A)).max() ** 2) for A in Aeq]; del Aeq
# the equalisation as a fifth null model, on the 100 connectomes of the other nulls (same input as the real network)
ip = [list(sid).index(s_) for s_ in pick]
eqrows = pd.DataFrame(dict(sid=pick.astype(int), model="InterEqualised", dMC=d.MC_eq.values[ip] - d.MC.values[ip],
                           dMC_pred=B_LOG * np.log(d.rs_eq.values[ip] / d.rs.values[ip])))
nulls = pd.concat([nulls[nulls.model != "InterEqualised"], eqrows]); nulls.to_csv(OUT / "nulls_sim.csv", index=False)
p = {m: stats.wilcoxon(g.dMC).pvalue for m, g in nulls.groupby("model")}
order = np.argsort(list(p.values())); padj = np.minimum.accumulate((np.array(list(p.values()))[order] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
pfdr = dict(zip(np.array(list(p))[order], np.minimum(padj, 1)))
N["nulls"] = {m: dict(n=len(g), dMC_median=float(g.dMC.median()), dMC_pred_median=float(g.dMC_pred.median()),
                      r_obs_pred=r_(g.dMC, g.dMC_pred), p=float(p[m]), p_fdr=float(pfdr[m])) for m, g in nulls.groupby("model")}
hsh = np.array([(mats[i][HM[ds[i]]].astype(float) ** 2).sum() / (mats[i].astype(float) ** 2).sum() for i in range(len(mats))]); H0 = float(hsh.mean())
Ah = []
for i in range(len(mats)):
    A = mats[i].astype(float); M = HM[ds[i]]; Hs = (A[M] ** 2).sum(); T2 = (A ** 2).sum()
    Ah.append(A * np.where(M, np.sqrt(H0 * (T2 - Hs) / (Hs * (1 - H0))), 1.0))
d["MC_hom_eq"] = simulate("hom_equalised", sid, np.array(Ah)).loc[sid].values; del Ah
iu = np.triu_indices(90, 1); ti, ta = [], []
for i in range(len(mats)):
    A = mats[i].astype(float); w = A[iu]; m_ = sets_of(ds[i])[2][iu]; T2 = (A ** 2).sum()
    ti.append(2 * w[m_].max() ** 2 / T2); ta.append(2 * w[~m_].max() ** 2 / T2)   # share of the squared weight on the strongest connection
d["top_inter"], d["top_intra"] = ti, ta
fr2 = np.array([(mats[i].astype(float) ** 2)[sets_of(ds[i])[2]].sum() / (mats[i].astype(float) ** 2).sum() for i in range(len(mats))])   # share of the squared weight between the hemispheres
d[["MC", "MC_eq", "MC_hom_eq", "top_inter", "top_intra"]].assign(frac=fr, frac2=fr2).to_csv(OUT / "lifespan_equalised.csv")
grp = pd.cut(d.age, [-1, 0, 5, 12, 21, 37, 50, 64, 75, 100])
N["dominance"] = dict(r_MC=dict(top_inter=r_(d.MC, d.top_inter), top_intra=r_(d.MC, d.top_intra)),
                      r_age_within={c: dict(top_inter=r_(g.age, g.top_inter), top_intra=r_(g.age, g.top_intra)) for c, g in d.groupby("cohort") if g.age.std() > 0},
                      by_age=d.groupby(grp, observed=True)[["top_inter", "top_intra"]].mean().round(4).reset_index().astype(str).to_dict(orient="list"))

rm = d.groupby("age").MC.mean().rolling(5, center=True, min_periods=3).mean()
W = {c: dict(n=len(g), MC=r_(g.age, g.MC), MC_eq=r_(g.age, g.MC_eq), MC_hom_eq=r_(g.age, g.MC_hom_eq)) for c, g in d.groupby("cohort") if g.age.std() > 0}
dev, agi = d[d.age <= 32], d[d.age > 32]
ag = np.where(d.cohort.isin(["camCAN", "HCPa"]).values & (d.age.values > 32))[0]; a = d.iloc[ag]
m = d[d.cohort.isin(["camCAN", "HCPa"]) & (d.age >= 36)]
Z = np.c_[np.ones(len(a)), a.hom_F2]; ra = lambda y: y - Z @ np.linalg.lstsq(Z, y, rcond=None)[0]
inter8 = sets_of(8)[2]; iu = np.triu_indices(90, 1)        # camCAN and HCPa share the AAL3 order
sh = np.array([np.where(inter8, 0, mats[i].astype(float))[iu].max() ** 2 * 2 / (mats[i].astype(float) ** 2).sum() for i in ag])
def within_r(g, col):                                   # correlation with age after removing the mean of each cohort
    g = g[g.groupby("cohort").age.transform("std") > 0]
    return r_(g.age - g.groupby("cohort").age.transform("mean"), g[col] - g.groupby("cohort")[col].transform("mean"))
N["interhemispheric"] = dict(F0=F0, H0=H0, within=W, running_mean_min_age=float(rm.idxmin()),
    within_regime={reg: {c: within_r(g, c) for c in ("MC", "MC_eq", "MC_hom_eq")} for reg, g in (("development", dev), ("aging", agi))},
    pooled={reg: dict(MC=r_(g.age, g.MC), MC_eq=r_(g.age, g.MC_eq), MC_hom_eq=r_(g.age, g.MC_hom_eq)) for reg, g in (("development", dev), ("aging", agi))},
    r_MC_h=dict(all=r_(d.hom_F2, d.MC), development=r_(dev.hom_F2, dev.MC), aging=r_(agi.hom_F2, agi.MC)),
    top_intra_share=dict(r_age={c: r_(a.age.values[a.cohort.values == c], sh[a.cohort.values == c]) for c in ("camCAN", "HCPa")},
                         young_33_44=float(sh[a.age.values <= 44].mean()), old_75_90=float(sh[a.age.values > 74].mean())))
N["aging_pooled"] = dict(n=len(a), r_age={c: r_(a.age, a[c]) for c in ("MC", "Ratio", "hom_F2", "inter_frac")} | {"cii": r_(a.age, pd.read_csv(OUT / "race_metrics.csv").set_index("sid").cii.loc[a.index].values)},
                         r_age_MC_given_h=r_(ra(a.age.values), ra(a.MC.values)), r_MC_h=r_(a.MC, a.hom_F2),
                         matched_36_89={c: dict(camCAN=float(m[m.cohort == "camCAN"][c].mean()), HCPa=float(m[m.cohort == "HCPa"][c].mean()),
                                                p=float(stats.ttest_ind(m[m.cohort == "camCAN"][c], m[m.cohort == "HCPa"][c]).pvalue)) for c in ("MC", "hom_F2", "Ratio")})
pd.DataFrame(W).T.round(2)

### Strongest connections at three stages of life (drawn in Fig. 5f)
Share of the squared weight of each connection, averaged over the subjects of each cohort and then over cohorts (equal weight per cohort), at birth, at 6-12 years (HCPd), at 30-40 years and from 76 years; region centroids from the AAL atlas shipped with `atlasreader`.

In [ ]:
def r_ci(a, b):                                        # correlation and its 95% confidence interval (Fisher z)
    r = r_(a, b); h = 1.96 / np.sqrt(len(a) - 3); return [r, float(np.tanh(np.arctanh(r) - h)), float(np.tanh(np.arctanh(r) + h))]
COLS = ("MC", "MC_eq", "MC_hom_eq")
N["interhemispheric"]["r_age_ci"] = {reg: {c: r_ci(g.age, g[c]) for c in COLS} for reg, g in (("development", d[d.age <= 32]), ("aging", d[d.age > 32]))}
N["interhemispheric"]["r_age_ci_within"] = {coh: {c: r_ci(g.age, g[c]) for c in COLS} for coh, g in d.groupby("cohort") if coh in ("camCAN", "HCPa")}

# Strongest connections at three stages of life, drawn in Fig. 5f (cohorts weighted equally within a stage)
XYF = OUT / "aal_centroids.csv"
if not XYF.exists():                    # region centroids from the AAL atlas shipped with atlasreader (MNI, mm; pip install atlasreader nibabel)
    import nibabel as nib, importlib.util
    P = Path(importlib.util.find_spec("atlasreader").origin).parent / "data/atlases"; img = nib.load(P / "atlas_aal.nii.gz"); L = pd.read_csv(P / "labels_aal.csv")
    a = np.asarray(img.dataobj).astype(int); xyz = nib.affines.apply_affine(img.affine, np.array(np.nonzero(a)).T); v = a[a > 0]
    C = {r.name: xyz[v == r.index].mean(0) for r in L.itertuples() if (v == r.index).any()}
    REN = {"Frontal_Sup": "Frontal_Sup_2", "Frontal_Mid": "Frontal_Mid_2", "Frontal_Inf_Orb": "Frontal_Inf_Orb_2", "Frontal_Sup_Orb": "OFCant", "Frontal_Mid_Orb": "OFClat"}
    key = lambda n: REN.get(n[:-2], n[:-2]).replace("Cingulum", "Cingulate") + n[-2:]
    pd.DataFrame([dict(node=i, aal90=AAL90[i], aal3=AAL3[i], x90=C[key(AAL90[i])][0], y90=C[key(AAL90[i])][1],
                       x3=C[key(AAL3[i])][0], y3=C[key(AAL3[i])][1]) for i in range(90)]).round(1).to_csv(XYF, index=False)
iu = np.triu_indices(90, 1); between = (iu[0] % 2) != (iu[1] % 2)         # left and right regions alternate in both orders
STG = [("newborns", d.age.values == 0), ("6-12 years", (d.cohort.values == "HCPd") & (d.age.values <= 12)),          # childhood: HCPd (typical development)
       ("30-40 years", (d.age.values >= 30) & (d.age.values <= 40)), ("76 years or more", d.age.values >= 76)]
rows, N["stages"] = [], {}
for name, msk in STG:
    idx = np.where(msk)[0]; coh = d.cohort.values[idx]; a3 = {ds[i] in AAL3_COHORTS for i in idx}; assert len(a3) == 1   # one order per stage
    Sh = np.array([mats[i].astype(float)[iu] ** 2 for i in idx]); Sh /= Sh.sum(1, keepdims=True)
    Sw = np.array([mats[i].astype(float)[iu] for i in idx]); Sw /= Sw.sum(1, keepdims=True)                     # share of the weight (not squared)
    ms = np.mean([Sh[coh == c].mean(0) for c in np.unique(coh)], axis=0); top = np.argsort(ms)[::-1]
    LAB = AAL3 if a3.pop() else AAL90
    rows += [dict(stage=name, i=int(iu[0][e]), j=int(iu[1][e]), share=float(ms[e]), between=bool(between[e]), order="aal3" if LAB is AAL3 else "aal90") for e in top[:15]]
    N["stages"][name] = dict(n=len(idx), cohorts={c: int((coh == c).sum()) for c in np.unique(coh)},
                             MC=float(np.mean([d.MC.values[idx][coh == c].mean() for c in np.unique(coh)])), between_share=float(ms[between].sum()),
                             strongest=f"{LAB[iu[0][top[0]]]}-{LAB[iu[1][top[0]]]}", strongest_share=float(ms[top[0]]),
                             strongest_is_top_in=float((Sh.argmax(1) == top[0]).mean()),
                             strongest_share_by_cohort={c: float(Sh[coh == c][:, top[0]].mean()) for c in np.unique(coh)},
                             between_share_by_cohort={c: float(Sh[coh == c][:, between].sum(1).mean()) for c in np.unique(coh)},
                             MC_by_cohort={c: float(d.MC.values[idx][coh == c].mean()) for c in np.unique(coh)},
                             rs=float(np.mean([d.rs.values[idx][coh == c].mean() for c in np.unique(coh)])),
                             rs_by_cohort={c: float(d.rs.values[idx][coh == c].mean()) for c in np.unique(coh)},
                             between_weight_share=float(np.mean([Sw[coh == c][:, between].sum(1).mean() for c in np.unique(coh)])),
                             between_weight_share_by_cohort={c: float(Sw[coh == c][:, between].sum(1).mean()) for c in np.unique(coh)})
pd.DataFrame(rows).to_csv(OUT / "stage_edges.csv", index=False)
# strongest connection of each network, by cohort (SI Appendix, Note 8)
rows_ = []
for c in ("dHCP", "BCP", "CALM", "RED", "ACE", "HCPd", "HCPya", "camCAN", "HCPa"):
    idx = np.where(d.cohort.values == c)[0]; LAB = AAL3 if ds[idx[0]] in AAL3_COHORTS else AAL90; top, v12 = [], []
    for i in idx:
        A = mats[i].astype(float); top.append(int(np.argmax(A[iu]))); lam, V = np.linalg.eigh(A)
        v12.append(float(np.sort(V[:, np.argmax(np.abs(lam))] ** 2)[-2:].sum()))     # leading eigenvector: weight on its two largest entries
    vals, cnt = np.unique(top, return_counts=True); e = vals[np.argmax(cnt)]; g = d.iloc[idx]
    rows_.append(dict(cohort=c, n=len(idx), order="AAL3" if LAB is AAL3 else "AAL90", strongest=f"{LAB[iu[0][e]]}--{LAB[iu[1][e]]}",
                      pct_strongest=100 * cnt.max() / len(idx), share=float(np.maximum(g.top_inter, g.top_intra).mean()),
                      v1_top2=float(np.mean(v12)), h=float(g.hom_F2.mean()), r_MC_h=r_(g.MC, g.hom_F2)))
pd.DataFrame(rows_).to_csv(OUT / "strongest_by_cohort.csv", index=False)
pd.DataFrame(N["stages"]).T[["n", "MC", "between_share", "strongest", "strongest_share", "strongest_is_top_in"]]

## 4b. CALM: referred children (simulated) vs comparison children of the same cohort

In [ ]:
import scipy.io as sio
allm = sio.loadmat(ALLMAT, squeeze_me=True, struct_as_record=False)["all_data"].connectomes.astype(float)
demo = pd.read_csv(DEMO); ref = np.where((demo.dataset.values == 3) & (demo.group.values == 2))[0]
Aref = np.array([allm[i] - np.diag(np.diag(allm[i])) for i in ref])
mc_ref = simulate("calm_referred", 10_000_000 + ref, Aref)
hm, hl, _ = sets_of(3); hom = np.vstack([hm, hl])
h_of = lambda A: 2 * (A[hom[:, 0], hom[:, 1]] ** 2).sum() / (A ** 2).sum()
cm = pd.concat([pd.DataFrame(dict(MC=mc_ref.values, rs=[rs_of(A) for A in Aref], h=[h_of(A) for A in Aref],
                                  age=demo.age.values[ref], sex=demo.sex.values[ref], referred=1)),
                d[d.cohort == "CALM"].assign(h=lambda t: t.hom_F2, referred=0)[["MC", "rs", "h", "age", "sex", "referred"]]], ignore_index=True)
cm.to_csv(OUT / "calm_sim.csv", index=False)
def group_effect(y):
    Xc = np.c_[np.ones(len(cm)), cm.referred, cm.age, cm.sex]; beta, *_ = np.linalg.lstsq(Xc, y, rcond=None)
    resid = y - Xc @ beta; dof = len(y) - Xc.shape[1]; se = np.sqrt(resid @ resid / dof * np.linalg.inv(Xc.T @ Xc)[1, 1])
    t = beta[1] / se; ra = y - np.c_[np.ones(len(cm)), cm.age, cm.sex] @ np.linalg.lstsq(np.c_[np.ones(len(cm)), cm.age, cm.sex], y, rcond=None)[0]
    g1, g0 = ra[cm.referred == 1], ra[cm.referred == 0]
    return dict(coef=float(beta[1]), t=float(t), p=float(2 * stats.t.sf(abs(t), dof)),
                d=float((g1.mean() - g0.mean()) / np.sqrt((g1.var(ddof=1) + g0.var(ddof=1)) / 2)))
N["calm"] = dict(n_referred=int(cm.referred.sum()), n_comparison=int((1 - cm.referred).sum()),
                 **{y: group_effect(cm[y].values if y != "rs" else np.log(cm[y].values)) for y in ("MC", "rs", "h")})
pd.DataFrame({k: v for k, v in N["calm"].items() if isinstance(v, dict)}).round(3)

## 5. Development, stated honestly

Within-cohort age trends (95% CI), the turning point of memory within the single-site camCAN cohort (piecewise-linear fit, breakpoint by grid search, 1,000 bootstrap resamples), and the young adults of HCPya vs camCAN at matched ages.

In [ ]:
N["within"] = {}
for c, g in d.groupby("cohort"):
    if g.age.std() == 0: continue
    g = g.join(race[["cii"]])
    for col in ("MC", "hom_F2", "Ratio", "rs", "cii"):
        y = np.log(g.rs) if col == "rs" else g[col]; r, pv = stats.pearsonr(g.age, y); z, se = np.arctanh(r), 1 / np.sqrt(len(g) - 3)
        N["within"][f"{c}|{col}"] = dict(r=float(r), lo=float(np.tanh(z - 1.96 * se)), hi=float(np.tanh(z + 1.96 * se)), p=float(pv), n=len(g))
    if c in ("camCAN", "HCPa"):
        Z = np.c_[np.ones(len(g)), g.hom_F2]; ra = lambda y: y - Z @ np.linalg.lstsq(Z, y, rcond=None)[0]
        N["within"][f"{c}|MC_given_h"] = dict(r=r_(ra(g.age.values), ra(g.MC.values)))
def seg(x, y, b):
    Xs = np.c_[np.ones_like(x), x, np.clip(x - b, 0, None)]; beta = np.linalg.lstsq(Xs, y, rcond=None)[0]; return ((y - Xs @ beta) ** 2).sum(), beta
grid = np.arange(24, 70.5, 0.5); rng = np.random.default_rng(0); cc = d[d.cohort == "camCAN"]; x = cc.age.values; N["camcan_turning_point"] = {}
for col in ("MC", "rs"):
    y = np.log(cc.rs.values) if col == "rs" else cc.MC.values
    best = grid[np.argmin([seg(x, y, b)[0] for b in grid])]; beta = seg(x, y, best)[1]; boot = []
    for _ in range(1000):
        i = rng.integers(0, len(x), len(x)); boot.append(grid[np.argmin([seg(x[i], y[i], b)[0] for b in grid])])
    early = x <= best; r, pv = stats.pearsonr(x[early], y[early])
    N["camcan_turning_point"][col] = dict(age=float(best), ci=[float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))],
                                          slope_before=float(beta[1]), slope_after=float(beta[1] + beta[2]), r_before=float(r), p_before=float(pv), n_before=int(early.sum()))
m = d[(d.age >= 22) & (d.age <= 37) & d.cohort.isin(["HCPya", "camCAN"])]
N["young_adults"] = {col: {c: float(g[col].mean()) for c, g in m.groupby("cohort")} | {"p": float(stats.ttest_ind(*[g[col] for _, g in m.groupby("cohort")]).pvalue)}
                     for col in ("MC", "hom_F2")}
N["camcan_turning_point"]

In [ ]:
json.dump(N, open(OUT / "numbers_p1.json", "w"), indent=1, default=float)
print("done ->", OUT)